# Trabajo final · Data Scientist · Marea

**Consultora macroeconómica** · Empresa ficticia. Los datos son sintéticos y **se generan por código en este notebook**: ningún resultado representa una situación real.

## La pregunta oficial

> **¿Qué inflación mensual esperar en los próximos tres meses, y con qué margen de error?**

**Decisión que habilita:** La inflación esperada y el rango que se les publica a los clientes.

**Tipo de problema:** Regresión de series de tiempo con pocos datos.

Las instrucciones completas (qué se entrega, criterios de calidad y defensa) están en el PDF **«Trabajo final · Data Scientist · Instrucciones»**. Este notebook es tu lugar de trabajo: vas de los datos sucios a la recomendación sin salir de acá.

| Sección | Módulo del programa |
| --- | --- |
| 1 | Introducción: definición del problema |
| 2 | Python: limpieza |
| 3 | Estadística |
| 4 | Machine Learning y regresión |
| 5 | Deep Learning |
| 6 | Apache Superset |
| 7 | De modelo a producción |
| 8 | Ciencia de datos aplicada al negocio |
| 9 | Fichas y recomendación |

## 0 · Preparación

Ejecutá esta celda una sola vez. `xgboost` es opcional: si querés usarlo, descomentá la última línea.

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib
# %pip install -q xgboost


## 1 · Datos y definición del problema

La celda siguiente **genera las tablas**. Tarda unos segundos. Después mirá qué hay con `inventario()` y `diccionario()`.

Antes de modelar, definí el problema: **tipo de problema, población, período, granularidad y variable objetivo**, y qué información está realmente disponible en el momento de decidir.

In [ ]:
import zlib, base64
TEXTO = (
    "eNqtWUlv40YWvvtXFDhogHTLbIqy5KWjXBIEyGHmMr4EgiCUyZLMMbewKLXdhue/zHEOOQxym6v/2HzvVXHT4qSBaTRarKpXb1+r"
    "k6wsqlrk26x8FlKLvByJUuYxPvG3jM/OYrUWG5WrSlarWNaFdrXKkjSV8zAIZ+OR2KkqlvH8J5lq5d2eCfypxByY/AqIiswHBrlN"
    "61WVb5q73mdxB5hwdvlZZEorjUUZ+6WqkiJe4d5GuU4YBOFFMHbAEe/r+d1IrCv169z5qwMMMk8K3OP7/rOSlb+T6VZpRtkeZEVe"
    "P9gTw5zaGPa+PKhKuYzlO/ASjCcjEYyOnlyPBCQNPVBN4idzXRo27zzG+hdRJ2UhYiUimd0nxS0+q2QnBfQrqrd/b5JM5eJZaJlC"
    "h6Lc5vVWppaluI76LDGHc+Il8Kejg31wMvZnIzHxwZD4KCo/L6pMpm4wOo7j+jSOkHAYCSxnc/ECzY8vSfO3giSn5fVFMMXypl1d"
    "0+GlWd6YZRjyMoTRrrA0N8PJxTjEanL9ylTWUMcjnEYkuaXoJ7XKtOvdkhoWaaJrN5Olq+tqZEzoeX6Sx+rJffSW4uNc7IY6i9Kk"
    "dLEaiQsY8G9Frow8ZWLOv6oKTnsH04He/lYE99jfU0/lan8PaluVw00mEjW+UMlndxH4kykbjf+9mi7p5kOxBzPtwcxCgtnIbA8m"
    "uOTjYMb/XjMmHe8BkcI7kstG7EWwBODYD1ji3ooEa5etNWoyhfFleMWdDWEbKatyUdMFt7LR7HriO5AbB544h+dt8wQ4MheczLz2"
    "InjgW9ECKPC5JC99KNrVOUOICzGmE0jfO3FJFvySM9R0HPiX7ZqveLw5DrBJ8lk8uGmQiO9FYECCGfHYCEE7IWFy4VW0AbFJ4slB"
    "EOm44edQpEw+ufzNhpl2AMwKQTCVj8KK0RfU0u8xPaRL5x1CtpbFODYYZ4cYJ6QGA/rHKMnbG5R8cYiLsBuQQ1Q++9pNmyp6sYcV"
    "nY1t8PUDiM9p2QdgFDKqyQ8D4gKAgHEJfpvpbQZqQTAOjQ3xyxA6yV37icg+5xz/SYxD70BiXCE/hgz4pismk88pk4fst+QB4vu5"
    "uGwXSPOzJhHWUsse+9cggDRJ2or9vyOpKw0P8Hz9kKxrd+z5a1S0XNJXVaRpgiqHPJQl+aopWzjJlMxdz5ahA46viV8uMZ2CoBFU"
    "Zhkl2NGnVdWzUBAEJmlcErpGGJXbtAHOdCkjRSQ50A0ZXW6rRJq8HxU5cBarSKFY1QUl8QDMji+JNGPCIvg2e8yMbE4NMhq4KxWp"
    "sk52hH3K2MMW+/k5MtOE9qYGc4RM+2eIXDGR1s+bP04pgbeWWULiaCpDRHHWF+eEe7FXwPDeUXpTK1SjsI0k5GSij+Kqhz08IQdj"
    "R4y9g92Uy3up1Soz5j8ZKpcmIbKLJoSR+dgPip5TaBXVRcVu9bJ2EL8Sjroyu6uXx1cryxFyFOyGgmdy9Y2JUpLz8UgoBhND1tT9"
    "QaWZWRE1hxT7X5KvU/b4FfxPoz8CI2UCPe/QfFJ/tTLNlcO9AvYt63oFAGxyJnLAMSSR8DGpVy1KHJtE5KAF1UwDkVXjurS3KfXh"
    "GJkp2SVoaXE3TiKFE2yRByMvrMoiTeokktilNbZLeDS0N4hXQ6xbt2Aa/r/ZpjJueeb6hHNj6YYlHJiNvo78bYkOXLmN+agnGBw0"
    "wezZRver3BQnFTueHlVscKhW6kVOa5Wy7UmdXh1X6DQ4os/JaWWGwTsanB7T3lWv3SS3476yVRwclTdahd1abS0eqTqi4UjW4tHX"
    "wFbrL0n94DY6cTyhMOggkRj84Gmr8shm0Mdb4TQa7mgbI73aOeHi4gKuc5/iBrIShJGYttB0bAmJQZqkjG6xbGXIi6xpmo3Fm6a5"
    "zXlgGEBU5Q6VsdsdTjw01kCrO0J7zunFY/bQ0cFUCMgYCkqQPDVIbt7+m0NuId/+U3QNStqjecTIQ6o0N30nxrNrQ5Ba3+6Lcgll"
    "nE/C3S0As7T7HmUZNKsecWZclEYsEhDKIxFmPXa0skR37eZeg4vu1mjXl2WJFO26rFgeM7inW6eFrN3dDosmWSZcQFH7f5S1/KmS"
    "mXIZxUhERbrNcj1fOGwUh3yUS36ximuHoytFQsVw7dhOMg+AKlW5C6SI3nVGTVCF8aZWG1VpSpqIggYAEMAHCCz8DjNjyvRgGHAw"
    "oRPFtbqnnwyBQZF3X5nVM/38Y5ubn5TPNgX9aFXSTxExw3mxox8YsOEYpBeNVM6yb1Bmfm4n5ugB47eMY7f/nekFbvpxPRjCubcE"
    "kQvH46vIAiyDBeVBXur6uVQ0AwImpu85fXujA/Jjfh3oX7BkRj3trJ2X7DYI49dPL8+vJjBh9Wfyi6/o8PpMGnSWD691gqjIJJvK"
    "TkGtiXgamkw/i/qpHlgENJ9u/cna0nsiYmTH1idopGPt8s5Qt0Su02ulypQaN5CA3nyy0IiUh3XfRQl7XBWl2/nlaXeE82H6lvlp"
    "oYLwc4d48U8DvkQrUz67hmy8LS1ZLbMyhU0qGc2p10BKNjhXSKG1msPB26cf73MXUuifIsTbAhsjQrf0BqjG76Ch3g0dNyK3Xpnn"
    "AZJ9fldtVc9z10jOtWLlVlBmgcyEYEEnXCf1tkahAWJOdKhAXEbyiNpf5GXJUYI2HqekRWfZC0ybysvtfYqbtkeHMKgiOy6E9mEH"
    "AN+ax4FhxZM+9VY2wdKkephhR7YUuTwfGtD3ShYM472TGAfdM8SmPErDIqXmw8aOUjNIWm4xcgsY6nQ1sJzuzCDex2Uw7CNjcM7C"
    "A7bW5HBuk659GLBOMrJhVroPxZc50mDseBZ78lW55CbwsztAxQoO7MbyWc9trQe3S29IAAYblgZklJYeQm6NSaGu1kTVdT7Enz5k"
    "nz784hjuhy8kUyPD8MIvFx+yiw8xBa+pNNAzDX6c6v+AtAvhjwlDQdErITc03l0HSFzeu5R3BmtoFTB05b16h8P3qp0pPEhVqx6W"
    "NuHwBzijveURYn5aRItBDhocN9lo3OKzmTKX+dGXVzQu5PV8lu5Snp2OjTL0gGgmGSg/TvY7rhPBYRPXPfXb9qH6uDt6XVtxyk9B"
    "98v+bO7fXNHTbGDbgJg7IIhxOhKDsAXtfPm+IntwFgnIOannIxTc9o3bvGFHx+6VhGAuGMZG5MbealLIxLxEja8PbtzYdzl+F2xC"
    "YZhwYmT5L1+asksLevVNeu2YgWAoK62V5HCeNQnIHrc5wzhG8/gR7XsyEeu7snFbUx7raFWsE3ThqV2VElVApUXjtutwv1rWkXVP"
    "agAii26vTYJ56shvCFFrcSyDvAfUxm4j1aLP64Dace4mh73EfqNyOWhUwEtHgdJEr+v4ppv2sQ8RelJ1ATUauEZpgAD3dY+L753S"
    "7NIZG5CDFqgzsLdozPO+qU0GiLDKY8oj+x70YtPfbTNSUhknDXWjIM4WvcGQZsq9WZD6X1OE+ilzRd5Jl7tZdP/eq+28ZPNfNVxf"
    "MQyjEWpme56YuaNyBpmUHlP6a5KcRv9Y2el/ZTInPW3Q40qng0Fev+0p59U+NNTbKjdMnZ39+PMPP7zLWqdBzBsZAhi5OxUtdL+w"
    "8DithUzFr1vk96ICrrJAt0fDKNHrqsyt/ThEZntAABTVBilLZwWjSwvA0f9eSOe1ezs80Nn/m93DSolr8dtvkiZauocsZk7ffhdK"
    "S8HF8+33nOmeENqiiwtC0r/Ur76ARurXIpWdIxBeNCEUChiIQjrjHlbGQ62c8pQXG1SNDA9v/7pP0r0YuzX3ezX6RPQdApLMmapY"
    "tBasz9hJL/0TdhuEbG810PTpOGWJNTEbq6yopKjffisBY951yOZRoipLmBzjeV/1zuvr2f8AILWKQw=="
)
exec(zlib.decompress(base64.b64decode(TEXTO)).decode())

datos = generar_datos()          # genera las tablas, con una semilla fija: todos los equipos de esta empresa ven lo mismo
globals().update(datos)          # cada tabla queda disponible como variable


In [ ]:
SLUG = "01_Marea"
import os, sqlite3
import pandas as pd

def inventario():
    """Tablas disponibles: filas, columnas y porcentaje de celdas faltantes."""
    return pd.DataFrame([(n, len(df), len(df.columns), round(float(df.isna().mean().mean() * 100), 1)) for n, df in datos.items()], columns=["tabla", "filas", "columnas", "% de faltantes"])

def diccionario(tabla=None):
    """Qué significa cada columna. Es una descripción neutra: no dice qué está limpio ni cuándo se conoce cada dato."""
    return pd.DataFrame([(t, c, d) for t, cols in DICC.items() if tabla in (None, t) for c, d in cols.items()], columns=["tabla", "columna", "qué es"])

bitacora_calidad = pd.DataFrame(columns=["problema", "dónde", "filas afectadas", "qué decidimos y por qué"])
def anotar(problema, donde, filas, decision):
    """Deja anotado un problema de calidad y la decisión tomada. Es parte de la trazabilidad."""
    bitacora_calidad.loc[len(bitacora_calidad)] = [problema, donde, filas, decision]; return bitacora_calidad

def exportar_superset(carpeta="superset", **tablas):
    """Guarda las tablas que pases como archivo SQLite y como CSV, para conectar Apache Superset."""
    os.makedirs(carpeta, exist_ok=True); con = sqlite3.connect(f"{carpeta}/{SLUG}.db")
    for nombre, df in tablas.items():
        df.to_csv(f"{carpeta}/{nombre}.csv", index=False); df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, con, index=False, if_exists="replace")
    con.close(); print(f"Listo: {carpeta}/{SLUG}.db y un CSV por tabla. Conectá Superset a ese archivo."); return carpeta


In [ ]:
inventario()

In [ ]:
diccionario()

### Tu definición del problema

Completá este diccionario. Si cambia más adelante, cambiá también la respuesta y anotá por qué.

In [ ]:
definicion = {
    "tipo de problema": "",
    "población": "",
    "período": "",
    "granularidad (una fila es…)": "",
    "variable objetivo": "",
    "qué se sabe en el momento de decidir": "",
}
definicion


## 2 · Python

**Qué tenés que resolver en tu caso:** Pasar los datos diarios a mensuales, unificar fechas, decimales y cambios de base de los índices, y decidir qué hacer con los meses faltantes.

Limpiá **sin borrar en silencio**: cada decisión se anota con `anotar(...)` y queda en la bitácora de calidad.

In [ ]:
# Tu trabajo


In [ ]:
bitacora_calidad

## 3 · Estadística

**Qué tenés que resolver en tu caso:** Estacionariedad, autocorrelación, quiebres de régimen y por qué dos series pueden parecer relacionadas sin estarlo.

Mostrá distribuciones e intervalos, y al menos una comparación o prueba con su conclusión y su límite.

In [ ]:
# Tu trabajo


## 4 · Modelado

**Qué tenés que resolver en tu caso:** Referencia autorregresiva, Ridge, Lasso, Random Forest y XGBoost; validación hacia adelante y error por horizonte.

Empezá por una referencia simple. Validá sin mirar el futuro y elegí la métrica por el costo del error, no por costumbre.

In [ ]:
# Tu trabajo


## 5 · Deep Learning

**Qué tenés que resolver en tu caso:** Una red contra la mejor referencia, con unos 180 meses de datos: ¿se justifica?

Entrená una red y compará contra la mejor referencia con la misma validación. Decidí si vale su costo y su opacidad.

In [ ]:
# Tu trabajo


## 6 · Superset

**Qué tenés que resolver en tu caso:** Tablero con el pronóstico, su banda de error y el error por horizonte.

Con `exportar_superset(nombre=tabla, ...)` llevás tus tablas a un archivo SQLite para conectar Apache Superset. Un gráfico por pregunta.

In [ ]:
# Tu trabajo


## 7 · Producción

**Qué tenés que resolver en tu caso:** Contrato de entrada, versión del modelo, reentrenamiento mensual y monitoreo del error.

Definí el contrato de entrada (campos, tipos y rangos), la versión, una prueba de humo y las señales de monitoreo con umbral, acción y responsable.

In [ ]:
# Tu trabajo


## 8 · Negocio

**Qué tenés que resolver en tu caso:** Cómo comunicar un número con su incertidumbre y qué escenarios mostrar; explicar sin confundir correlación con causa.

Decisión, costos, explicación global y local, y una recomendación proporcional a la evidencia.

In [ ]:
# Tu trabajo


## 9 · Fichas y recomendación

### Ficha de evidencia

| Campo | Tu respuesta |
| --- | --- |
| Decisión que se toma | |
| Población y período | |
| Procedimiento | |
| Resultado | |
| Validación | |
| Límite | |

### Ficha de modelo y plan de monitoreo

| Campo | Tu respuesta |
| --- | --- |
| Qué modelo se propone y por qué ese | |
| Con qué datos y qué versión | |
| Contrato de entrada | |
| Riesgos principales | |
| Señales a vigilar, umbral, acción y responsable | |

### Recomendación (una página)

- **Evidencia:** qué muestran los datos.
- **Interpretación:** qué creemos que significa, sin presentarlo como causalidad.
- **Recomendación:** qué conviene hacer, proporcional a la evidencia.
- **Una limitación y un siguiente paso.**